In [ ]:
# Library imports
import jax
import jax.numpy as jnp
import equinox as eqx  # NNs and other useful bits --> https://docs.kidger.site/equinox/

# Neural Operator Training

In this notebook, we will solve the 2D heat equation described previously using a DeepONet model. In this notebook we will learn the following:

##### TODO Add learnings

### Recap: The Heat Equation

The heat equation, as described in our first example, will be solved for a 2D domain, $\Omega = (0, 10) \times (0, 10)$ and $t > 0$. Recall that this transient PDE is given as:
\begin{equation}
    \tag{1}
    \frac{\partial u}{\partial t} - \alpha (\frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2}) = 0 \quad x \in [0, 100] \quad y \in [0, 100] \quad t \in [0, 20]
\end{equation}
where $u [K]$ is the temperature, $x, y [m]$ are the spatial coordinates, $t [s]$ is time, and $\alpha$ is the thermal diffusivity of the domain.

It has Dirichlet BCs on the bottom, top and left sides given as:
$$
    u(x, 0, t) = u(x, 1, t) = u(0, y, t) = 0 K
$$

With the right side given as:
$$
    u(1, y, t) = 100 K
$$

It has an IC inside throughout the domain given by:
$$
    u(x, y, 0) = 0 K \quad x \in [0, 100] \quad y \in [0, 100]
$$

### Training our DeepONet

Following our discussion in the [Neural-Operators notebook](ReCoDE-Neural-Operators/notebooks/02-Intro-to-Neural-Operators.ipynb), a standard DeepONet is trained using pairwise samples of input and output functions, $\{a^{(i)}, u^{(i)}\}^{N}_{i}$. The main steps involved in training is described as follows:

- $N$ representative input functions, $a^{(i)}$, are selected at random from the set $1 \le i \le N$. These functions are then evaluated at $m$ sensor locations, i.e. $a_{j}^{(i)} = a^{(i)}(x_{j})$ from the set $1 \le j \le m$.
- For each $a^{(i)}$, the corresponding solution functions, $u^{(i)}$, is determined (here using finite differences for the heat equation).
- We the sample $u^{(i)}$ at $R$ random locations, i.e. $u_{k}^{(i)} = u^{(i)}(y_{k})$ from the set $1 \le k \le R$.
- The training set, $S$, can then be built such that:
\begin{equation}
    \tag{2}
    S = { (a_{j}^{(i)}, y^{(k)}, u_{k}^{(i)})} \quad 1 \le i \le N, 1 \le j \le M, 1 \le k \le R
\end{equation}
which will contain $N \times R$ training samples.
- The loss function can then be defined to determine the differences between the predictions, $\hat{u}_{k}^{(i)}$, with the true value, $u_{k}^{(i)}$.
- Training is then performed to minimized this loss function and find the optimal parameters, $\theta$ for our DeepONet model.

### Our Datasets

The input function, $a$, for our NO will be based on different $\alpha$ values. We will generate multiple heat PDE solutions, that vary based on a constant $\alpha$ throughout the 2D domain. This is done in the [Data-Generation notebook](ReCoDE-Neural-Operators/notebooks/03-Dataset-Generation.ipynb).

### Defining our DeepONet

In [ ]:
class DeepONet2d(eqx.Module):
    """2D DeepONet model definition where the Branch & Trunk nets are configured as MLPs"""

    branch_net: eqx.nn.MLP
    trunk_net: eqx.nn.MLP
    bias: jax.Array

    def __init__(
        self,
        in_size_branch,
        in_size_trunk,
        out_size,
        width_size,
        depth,
        activation,
        *,
        key,
    ):
        b_key, t_key = jax.random.split(key, num=2)
        self.branch_net = eqx.nn.MLP(
            in_size=in_size_branch,
            out_size=out_size,
            width_size=width_size,
            depth=depth,
            activation=activation,
            key=b_key,
        )
        self.trunk_net = eqx.nn.MLP(
            in_size=in_size_trunk,
            out_size=out_size,
            width_size=width_size,
            depth=depth,
            activation=activation,
            final_activation=activation,
            key=t_key,
        )
        self.bias = jnp.zeros((out_size,))

    def __call__(self, x_branch, x_trunk):
        """
        x_branch.shape = (in_size_branch,)
        x_trunk.shape = (in_size_trunk,)

        return shape: "scalar"
        """
        branch_out = self.branch_net(x_branch)
        trunk_out = self.trunk_net(x_trunk)
        inner_product = jnp.sum(branch_out * trunk_out, keepdims=True)

        return inner_product + self.bias